# 🚀 ReqGPT / ThinkingPods — 100% Free Cloud Runner

Run ThinkingPods completely in the cloud on **Google Cloud's free 12.7 GB RAM & GPU infrastructure** without using your local computer's resources.

### ⚡ Quick Start:
1. Make sure you are connected to Colab (Click **Connect** in the top-right corner).
2. Click **Runtime ➡️ Run all** in the top menu.
3. Scroll to the bottom cell to see your generated Cloudflare backend URL.
4. Open the permanent web app at **[ThinkingPods Live Web App](https://bezaleelpaul.github.io/ThinkingPods-Live/)** and paste the backend URL to start chatting!

In [ ]:
# Step 1: Install lightweight dependencies & clone repo
!apt-get update -qq && apt-get install -y -qq ffmpeg curl libportaudio2
!pip install -q fastapi uvicorn streamlit faster-whisper ollama python-multipart

import os
if not os.path.exists("/content/ThinkingPods"):
    !git clone https://github.com/BezaleelPaul/ThinkingPods-Live.git /content/ThinkingPods
else:
    !git -C /content/ThinkingPods pull

%cd /content/ThinkingPods
print("✅ Environment & Codebase ready!")

In [ ]:
# Step 2: Install and Start Ollama with llama3.2:1b
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > /content/ollama.log 2>&1 &

import time, urllib.request
print("Waiting for Ollama service to start...")
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags")
        break
    except Exception:
        time.sleep(1)

!ollama pull llama3.2:1b
print("✅ Ollama and llama3.2:1b ready on Google Cloud!")

In [ ]:
# Step 3: Install Cloudflare Tunnel for Free Public URL
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared
print("✅ Cloudflared tunnel ready!")

In [ ]:
# Step 4: Start Backend and Cloudflare Tunnel
import os, time, re, urllib.request

# Start FastAPI backend in background
print("Starting FastAPI backend...")
!nohup python server.py > /content/server.log 2>&1 &

# Wait for backend health check
print("Waiting for backend to be ready...")
for _ in range(35):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health")
        break
    except Exception:
        time.sleep(1)
print("✅ Backend is live on port 8000!")

# Start Cloudflare Tunnel in background
print("Generating free public cloud URL...")
!nohup /content/cloudflared tunnel --url http://127.0.0.1:8000 > /content/tunnel.log 2>&1 &

backend_url = None
for _ in range(30):
    time.sleep(1)
    if os.path.exists("/content/tunnel.log"):
        with open("/content/tunnel.log", "r") as f:
            content = f.read()
            match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
            if match:
                backend_url = match.group(0)
                break

if backend_url:
    print("\n" + "="*65)
    print("🎉 ThinkingPods Backend is LIVE on Google Cloud!")
    print(f"🔗 Cloud Backend API URL: {backend_url}")
    print(f"👉 Open your Web App at: https://bezaleelpaul.github.io/ThinkingPods-Live/")
    print("="*65 + "\n")
else:
    print("Tunnel log:")
    if os.path.exists("/content/tunnel.log"):
        with open("/content/tunnel.log") as f:
            print(f.read())